# DNB v3 — Offline Smoke Tests

Interactive validation of the rebuilt pipeline architecture:

```
WaveletConvolution → TargetWaveDetector (activation) → AmplitudeMonitor (inhibition) → StimTrigger
```

This mirrors the Rust architecture: **Filter → Detector → Trigger**, where:
- The wavelet convolution replaces bandpass filter banks (single-pass, all bands)
- `TargetWaveDetector` = activation detector ("phase is at target")
- `AmplitudeMonitor` = inhibition detector ("HF amplitude too high")
- `StimTrigger` = pulse trigger (combines both with cooldowns)

In [ ]:
import sys
sys.path.insert(0, 'src')  # if running from repo root

import numpy as np
import matplotlib.pyplot as plt
from math import pi

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger
from dnb.validation.synthetic import (
    generate_synthetic_recording, save_synthetic, generate_pink_noise,
    inject_slow_wave, inject_ied,
)
from dnb.validation.ground_truth import validate, Annotation

import dnb
print(f'DNB v{dnb.__version__}')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

---
## 1. Clean sine wave — verify phase detection

Before adding noise, confirm the wavelet + detector correctly identifies
the target phase of a known waveform.

In [ ]:
# Generate a clean 1 Hz sine wave
fs = 1000.0
dur = 30.0
n = int(dur * fs)
t = np.arange(n) / fs
signal = 500.0 * np.sin(2 * pi * 1.0 * t).reshape(1, -1)

path = save_synthetic('/tmp/clean_sine.npz', signal, fs)

# Target phase = pi (descending zero crossing of the sine)
TARGET_PHASE = pi

pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=5),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 2.0),
            target_phase=TARGET_PHASE,
            phase_tolerance=0.2,
            amp_min=10.0, amp_max=100000.0,
            warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id=None,
            backoff_s=1.5,
        ),
    ],
    config=PipelineConfig(sample_rate=fs, n_channels=1, chunk_duration=0.5),
)

events = pipeline.run_offline()
stim1 = [e for e in events if e.event_type == EventType.STIM1]

print(f'{len(stim1)} STIM1 events')
if stim1:
    phases = np.array([e.metadata['phase'] for e in stim1])
    times = np.array([e.timestamp for e in stim1])
    print(f'Phase: mean={np.mean(phases):.3f} std={np.std(phases):.3f} (target={TARGET_PHASE:.3f})')

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Signal + stim markers
    axes[0].plot(t, signal[0], 'b-', lw=0.5, alpha=0.7)
    for e in stim1:
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1)
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].set_title('Clean 1 Hz sine + STIM1 markers')

    # Phase at detection
    axes[1].scatter(times, phases, c='red', s=20, zorder=5)
    axes[1].axhline(TARGET_PHASE, color='green', ls='--', label=f'target={TARGET_PHASE:.2f}')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

---
## 2. Synthetic slow waves with noise — detection accuracy

Generate a recording with planted slow waves at known times in pink noise.
Run the full pipeline and validate against ground truth.

In [ ]:
signal, gt_events, actual_snr = generate_synthetic_recording(
    n_channels=1, duration_s=120.0, sample_rate=1000.0,
    n_slow_waves=15, n_ieds=0,  # no IEDs for now
    snr=5.0, sw_amplitude=500.0, sw_frequency=1.0,
    seed=42,
)

path = save_synthetic('/tmp/synth_sw.npz', signal, 1000.0, gt_events)

sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Signal: {signal.shape}, SNR={actual_snr:.1f}')
print(f'Planted: {len(sw_gt)} slow waves')
print(f'SW times: {[f"{e.timestamp:.1f}s" for e in sw_gt]}')

# Plot the signal with SW markers
t = np.arange(signal.shape[1]) / 1000.0
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, signal[0], 'b-', lw=0.3, alpha=0.6)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.5, lw=1.5, label='planted SW' if e == sw_gt[0] else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Synthetic recording (SNR={actual_snr:.1f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Run detection pipeline (no inhibition — no IEDs planted)
pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 2.0),
            target_phase=pi,
            phase_tolerance=0.3,
            amp_min=50.0, amp_max=10000.0,
            warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id=None,
            backoff_s=3.0,
        ),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
)

detections = pipeline.run_offline()
stim1 = [e for e in detections if e.event_type == EventType.STIM1]
stim2 = [e for e in detections if e.event_type == EventType.STIM2]
print(f'STIM1: {len(stim1)}, STIM2: {len(stim2)}')

# Validate
annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(stim1, annotations, time_tolerance=0.5, target_type='SW')
print(report.summary())

# Plot detections vs ground truth
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, signal[0], 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2, label='GT' if e == sw_gt[0] else '')
for e in stim1:
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--', label='STIM1' if e == stim1[0] else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Detections (red) vs Ground Truth (green)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Add IEDs — test inhibition

Add interictal epileptiform discharges and verify the `AmplitudeMonitor`
correctly blocks stimulation during IEDs.

In [ ]:
signal_ied, gt_events_ied, snr_ied = generate_synthetic_recording(
    n_channels=1, duration_s=120.0, sample_rate=1000.0,
    n_slow_waves=15, n_ieds=8,
    snr=5.0, sw_amplitude=500.0, ied_amplitude=3000.0,
    seed=42,
)

path_ied = save_synthetic('/tmp/synth_sw_ied.npz', signal_ied, 1000.0, gt_events_ied)

sw_gt_ied = [e for e in gt_events_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_events_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

# Run WITH inhibition
pipeline_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 2.0),
            target_phase=pi, phase_tolerance=0.3,
            amp_min=50.0, amp_max=10000.0, warmup_chunks=3,
        ),
        AmplitudeMonitor(
            id='ied_monitor', freq_range=(10.0, 40.0),
            ref_freq_range=(0.5, 2.0), ratio_max=0.5,
            warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id='ied_monitor',
            backoff_s=3.0, inhibition_cooldown_s=3.0,
        ),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
)

det_inh = pipeline_inh.run_offline()
stim1_inh = [e for e in det_inh if e.event_type == EventType.STIM1]

# Run WITHOUT inhibition for comparison
pipeline_no_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 2.0),
            target_phase=pi, phase_tolerance=0.3,
            amp_min=50.0, amp_max=10000.0, warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id=None,
            backoff_s=3.0,
        ),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
)

det_no_inh = pipeline_no_inh.run_offline()
stim1_no_inh = [e for e in det_no_inh if e.event_type == EventType.STIM1]

print(f'\nWith inhibition:    {len(stim1_inh)} STIM1')
print(f'Without inhibition: {len(stim1_no_inh)} STIM1')

# Check: did any stims fire near IEDs?
def stims_near_ieds(stim_events, ied_events, window_s=1.0):
    count = 0
    for s in stim_events:
        for ied in ied_events:
            if abs(s.timestamp - ied.timestamp) < window_s:
                count += 1
                break
    return count

near_inh = stims_near_ieds(stim1_inh, ied_gt)
near_no_inh = stims_near_ieds(stim1_no_inh, ied_gt)
print(f'\nSTIMs within 1s of IED — with inhibition: {near_inh}, without: {near_no_inh}')

In [ ]:
# Visualise
t_ied = np.arange(signal_ied.shape[1]) / 1000.0
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, stims, label in [
    (axes[0], stim1_no_inh, 'WITHOUT inhibition'),
    (axes[1], stim1_inh, 'WITH inhibition'),
]:
    ax.plot(t_ied, signal_ied[0], 'b-', lw=0.3, alpha=0.5)
    for e in sw_gt_ied:
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2)
    for e in ied_gt:
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3,
                   label='IED' if e == ied_gt[0] else '')
    for e in stims:
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--')
    ax.set_ylabel('Amp')
    ax.set_title(label)

axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

---
## 4. SNR sweep — detection performance vs noise

Sweep across SNR levels to characterise how detection degrades with noise.

In [ ]:
snr_levels = [1.0, 2.0, 3.0, 5.0, 10.0]
results = []

for snr_target in snr_levels:
    sig, gt, actual = generate_synthetic_recording(
        n_channels=1, duration_s=120.0, sample_rate=1000.0,
        n_slow_waves=15, n_ieds=0, snr=snr_target,
        seed=int(snr_target * 1000),
    )
    p = save_synthetic(f'/tmp/snr_{snr_target}.npz', sig, 1000.0, gt)

    pipe = Pipeline(
        source=FileSource(str(p)),
        modules=[
            WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
            TargetWaveDetector(
                id='slow_wave', freq_range=(0.5, 2.0),
                target_phase=pi, phase_tolerance=0.3,
                amp_min=50.0, amp_max=10000.0, warmup_chunks=3,
            ),
            StimTrigger(
                activation_detector_id='slow_wave',
                inhibition_detector_id=None,
                backoff_s=3.0,
            ),
        ],
        config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
    )

    dets = pipe.run_offline()
    stim1 = [e for e in dets if e.event_type == EventType.STIM1]

    sw_gt = [e for e in gt if e.metadata.get('type') == 'SW']
    anns = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
    report = validate(stim1, anns, time_tolerance=0.5)

    results.append({
        'snr': actual,
        'n_planted': len(sw_gt),
        'n_detected': len(stim1),
        **report.metrics,
    })
    print(f'SNR={actual:.1f}: P={report.metrics["precision"]:.2f} '
          f'R={report.metrics["recall"]:.2f} F1={report.metrics["f1"]:.2f} '
          f'(TP={report.metrics["true_positives"]:.0f} '
          f'FP={report.metrics["false_positives"]:.0f} '
          f'FN={report.metrics["false_negatives"]:.0f})')

In [ ]:
snrs = [r['snr'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision / Recall / F1
axes[0].plot(snrs, [r['precision'] for r in results], 'o-', label='Precision', lw=2)
axes[0].plot(snrs, [r['recall'] for r in results], 's-', label='Recall', lw=2)
axes[0].plot(snrs, [r['f1'] for r in results], '^-', label='F1', lw=2)
axes[0].set_xlabel('SNR')
axes[0].set_ylabel('Score')
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title('Detection Performance vs SNR')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Detection counts
x = np.arange(len(snrs))
w = 0.25
axes[1].bar(x - w, [r['true_positives'] for r in results], w, label='TP')
axes[1].bar(x, [r['false_positives'] for r in results], w, label='FP')
axes[1].bar(x + w, [r['false_negatives'] for r in results], w, label='FN')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s:.0f}' for s in snrs])
axes[1].set_xlabel('SNR')
axes[1].set_ylabel('Count')
axes[1].set_title('Detection Breakdown')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 5. Inspect wavelet output

Look at the raw wavelet decomposition to debug detection parameters.

In [ ]:
# Run pipeline but capture intermediate results
from dnb.modules.base import ProcessResult
from dnb.core.ring_buffer import RingBuffer

# Generate a short segment for inspection
sig_short, gt_short, _ = generate_synthetic_recording(
    n_channels=1, duration_s=20.0, sample_rate=1000.0,
    n_slow_waves=3, n_ieds=1, snr=5.0, seed=123,
)

# Process manually to inspect intermediate state
config = PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=2.0)  # big chunks for viz
wavelet_mod = WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10)
wavelet_mod.configure(config)

ring = RingBuffer(n_channels=1, capacity=config.buffer_samples)

# Process first chunk
chunk_samples = config.chunk_samples
from dnb.core.types import DataChunk

chunk_data = sig_short[:, :chunk_samples]
ring.write(chunk_data)
chunk = DataChunk(
    samples=chunk_data,
    timestamps=np.arange(chunk_samples) / 1000.0,
    channel_ids=np.array([0], dtype=np.int32),
    sample_rate=1000.0,
)
result = ProcessResult(chunk=chunk, ring_buffer=ring)
result = wavelet_mod.process(result)

# Plot time-frequency representation
wav = result.wavelet
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

t_chunk = chunk.timestamps

axes[0].plot(t_chunk, chunk.samples[0], 'b-', lw=0.5)
axes[0].set_ylabel('Raw signal')
axes[0].set_title('Wavelet decomposition of first chunk')

# Amplitude spectrogram
im = axes[1].pcolormesh(
    t_chunk, wav.frequencies, wav.amplitude[0],
    shading='auto', cmap='hot'
)
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_yscale('log')
plt.colorbar(im, ax=axes[1], label='Amplitude')

# Phase in SO band
so_mask = (wav.frequencies >= 0.5) & (wav.frequencies <= 2.0)
so_phase = np.mean(wav.phase[0, so_mask, :], axis=0)
axes[2].plot(t_chunk, so_phase, 'g-', lw=0.5)
axes[2].axhline(pi, color='red', ls='--', alpha=0.5, label='target phase (π)')
axes[2].set_ylabel('SO Phase (rad)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 6. Parameter exploration

Experiment with detection parameters to understand their effect.

In [ ]:
# Sweep phase tolerance and backoff to see their effects
sig_pe, gt_pe, _ = generate_synthetic_recording(
    n_channels=1, duration_s=120.0, sample_rate=1000.0,
    n_slow_waves=15, n_ieds=0, snr=5.0, seed=42,
)
path_pe = save_synthetic('/tmp/param_explore.npz', sig_pe, 1000.0, gt_pe)
sw_gt_pe = [e for e in gt_pe if e.metadata.get('type') == 'SW']
anns_pe = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt_pe]

tolerances = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
tol_results = []

for tol in tolerances:
    pipe = Pipeline(
        source=FileSource(str(path_pe)),
        modules=[
            WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
            TargetWaveDetector(
                id='slow_wave', freq_range=(0.5, 2.0),
                target_phase=pi, phase_tolerance=tol,
                amp_min=50.0, amp_max=10000.0, warmup_chunks=3,
            ),
            StimTrigger(
                activation_detector_id='slow_wave',
                inhibition_detector_id=None, backoff_s=3.0,
            ),
        ],
        config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
    )
    dets = pipe.run_offline()
    s1 = [e for e in dets if e.event_type == EventType.STIM1]
    r = validate(s1, anns_pe, time_tolerance=0.5)
    tol_results.append({'tolerance': tol, **r.metrics, 'n_detected': len(s1)})
    print(f'tol={tol:.2f}: {len(s1)} STIM1, P={r.metrics["precision"]:.2f} R={r.metrics["recall"]:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tolerances, [r['precision'] for r in tol_results], 'o-', label='Precision')
ax.plot(tolerances, [r['recall'] for r in tol_results], 's-', label='Recall')
ax.plot(tolerances, [r['f1'] for r in tol_results], '^-', label='F1')
ax.set_xlabel('Phase tolerance (rad)')
ax.set_ylabel('Score')
ax.set_title('Effect of phase tolerance on detection')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Notes

**Architecture** (Rust-inspired):
- `WaveletConvolution` = filter (replaces bandpass banks)
- `TargetWaveDetector` = activation detector ("phase at target")
- `AmplitudeMonitor` = inhibition detector ("HF too high")
- `StimTrigger` = pulse trigger (combines both, applies cooldowns)

**Next steps**:
- Tune phase offset (wavelet group delay causes ~0.2 rad shift)
- Test with real NPlay data
- Add Downsampler for 30kHz hardware rate
- Ground truth validation with expert annotations